# Microsoft Foundry CSV Analysis Demo

This notebook demonstrates a **workshop-ready** pattern for:

1. downloading the two sample CSVs from Azure Blob Storage,
2. previewing them locally with pandas,
3. uploading them to **Microsoft Foundry**,
4. creating a **Code Interpreter** agent, and
5. asking the agent to analyze the files and optionally create charts.

## Datasets used

- `synthetic_mental_health_dataset.csv`
- `synthetic_coffee_health_10000.csv`

## Why we download locally first

The Microsoft Learn documentation for Foundry Code Interpreter says the sandbox can work with **attached files** and also says the sandbox **cannot make outbound network requests**. So the client notebook downloads the CSVs first, then uploads them into the agent session.

## Prerequisites

- A Microsoft Foundry project endpoint
- A deployed model that supports Code Interpreter in your region
- `az login` completed **or** another working Azure credential flow
- Python packages:
  - `azure-ai-projects>=2.0.0`
  - `azure-identity`
  - `pandas`, `matplotlib`, `seaborn`, `requests`

In [ ]:
# If needed, uncomment the next line to install requirements in a fresh notebook/kernel.
# %pip install --quiet "azure-ai-projects>=2.0.0" azure-identity pandas matplotlib seaborn requests python-dotenv

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool, AutoCodeInterpreterToolParam

## Configuration

Set your Foundry project endpoint and model deployment name here.

Example endpoint format from Microsoft Learn:

`https://<resource>.services.ai.azure.com/api/projects/<project-name>`

In [ ]:
PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT", "https://YOUR_RESOURCE.services.ai.azure.com/api/projects/YOUR_PROJECT")
MODEL_DEPLOYMENT = os.getenv("MODEL_DEPLOYMENT", "gpt-5-mini")

GENERAL_URL = "https://stwkshophiv4o.blob.core.windows.net/coffeecsv/GeneralHealth/synthetic_mental_health_dataset.csv"
COFFEE_URL = "https://stwkshophiv4o.blob.core.windows.net/coffeecsv/mentalHealth/synthetic_coffee_health_10000.csv"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
GENERAL_LOCAL = DATA_DIR / "synthetic_mental_health_dataset.csv"
COFFEE_LOCAL = DATA_DIR / "synthetic_coffee_health_10000.csv"

In [ ]:
def download_file(url: str, dest: Path) -> Path:
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"Downloaded {dest.name} -> {dest.resolve()} ({dest.stat().st_size:,} bytes)")
    return dest

# Download the workshop datasets from Blob Storage.
download_file(GENERAL_URL, GENERAL_LOCAL)
download_file(COFFEE_URL, COFFEE_LOCAL)

## Quick local inspection with pandas

This is useful in a workshop because it proves the files are reachable and helps participants see the schema before the agent starts reasoning over the data.

In [ ]:
general_df = pd.read_csv(GENERAL_LOCAL)
coffee_df = pd.read_csv(COFFEE_LOCAL)

print("General mental health shape:", general_df.shape)
print("Coffee health shape:", coffee_df.shape)

In [ ]:
print("General mental health columns:")
print(list(general_df.columns))
print("
Coffee health columns:")
print(list(coffee_df.columns))

In [ ]:
display(general_df.head())
display(coffee_df.head())

## Optional local baseline charts

These are not required for Foundry, but they give you a fallback live demo if the cloud tool is delayed by environment setup.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if {"Coffee_Intake", "Sleep_Hours"}.issubset(coffee_df.columns):
    sns.scatterplot(data=coffee_df, x="Coffee_Intake", y="Sleep_Hours", hue="Stress_Level", alpha=0.4, ax=axes[0])
    axes[0].set_title("Coffee Intake vs Sleep Hours")

if {"Growing_Stress", "Mood_Swings"}.issubset(general_df.columns):
    stress_mood = pd.crosstab(general_df["Growing_Stress"], general_df["Mood_Swings"])
    stress_mood.plot(kind="bar", stacked=True, ax=axes[1])
    axes[1].set_title("Growing Stress vs Mood Swings")
    axes[1].legend(title="Mood_Swings", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

# Create the Foundry Code Interpreter agent

The Microsoft Learn sample shows this pattern:
- create `AIProjectClient`
- call `project.get_openai_client()`
- upload CSV files with `purpose="assistants"`
- create an agent version with `CodeInterpreterTool(container=AutoCodeInterpreterToolParam(file_ids=[...]))`

In [ ]:
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
openai = project.get_openai_client()

In [ ]:
with open(GENERAL_LOCAL, "rb") as f:
    general_file = openai.files.create(purpose="assistants", file=f)

with open(COFFEE_LOCAL, "rb") as f:
    coffee_file = openai.files.create(purpose="assistants", file=f)

print("Uploaded general mental health file:", general_file.id)
print("Uploaded coffee health file:", coffee_file.id)

In [ ]:
AGENT_INSTRUCTIONS = """
You are a rigorous data-analysis agent for a scientific workshop.

You have two attached CSV files:
1. synthetic_mental_health_dataset.csv
2. synthetic_coffee_health_10000.csv

Rules:
- Always inspect schema first.
- Use Python for any numeric or plotting task.
- Explain your method before interpretation.
- Do not claim causation from correlation.
- If the user asks to compare the two datasets, perform a thematic comparison unless there is a clear join key.
- Save charts as PNG when generating files.
"""

agent = project.agents.create_version(
    agent_name="mental-health-research-agent",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=AGENT_INSTRUCTIONS,
        tools=[
            CodeInterpreterTool(
                container=AutoCodeInterpreterToolParam(
                    file_ids=[general_file.id, coffee_file.id]
                )
            )
        ],
    ),
    description="Workshop agent for mental health and coffee-health CSV analysis.",
)

print("Agent created:", agent.name, "version:", agent.version)

## Start a conversation and ask analysis questions

In [ ]:
conversation = openai.conversations.create()
print("Conversation ID:", conversation.id)

In [ ]:
def ask_agent(prompt: str):
    response = openai.responses.create(
        conversation=conversation.id,
        input=prompt,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
    )
    return response

In [ ]:
prompt_1 = """
Profile both CSV files. For each dataset, list the columns, infer likely variable types,
identify candidate target variables, and propose three useful workshop questions.
"""

response_1 = ask_agent(prompt_1)
response_1

In [ ]:
prompt_2 = """
Using synthetic_coffee_health_10000.csv, analyze the relationship between Coffee_Intake,
Sleep_Hours, and Stress_Level. Use Python. Provide summary statistics, at least one chart,
and a plain-English interpretation.
"""

response_2 = ask_agent(prompt_2)
response_2

In [ ]:
prompt_3 = """
Using synthetic_mental_health_dataset.csv, analyze whether Growing_Stress appears to be associated with
Mood_Swings, Changes_Habits, Social_Weakness, and Work_Interest.
Use contingency tables and at least one visualization.
"""

response_3 = ask_agent(prompt_3)
response_3

In [ ]:
prompt_4 = """
Compare the two datasets at a thematic level.
Do NOT invent a row-level join.
Discuss how lifestyle, sleep, stress, and work-related factors appear across the two datasets,
and suggest a small multi-agent workshop storyline based on these files.
"""

response_4 = ask_agent(prompt_4)
response_4

## Helper: extract generated files from response annotations

The Learn sample shows that generated files are returned as `container_file_citation` annotations.
This helper downloads any chart or file created by the Code Interpreter.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def download_generated_files(response, output_dir: Path = OUTPUT_DIR):
    saved = []
    for item in getattr(response, "output", []):
        if getattr(item, "type", None) != "message":
            continue
        for content in getattr(item, "content", []):
            if getattr(content, "type", None) != "output_text":
                continue
            for ann in getattr(content, "annotations", []) or []:
                if getattr(ann, "type", None) == "container_file_citation":
                    file_id = ann.file_id
                    container_id = ann.container_id
                    filename = ann.filename or f"{file_id}.bin"
                    file_content = openai.containers.files.content.retrieve(
                        file_id=file_id,
                        container_id=container_id,
                    )
                    out_path = output_dir / filename
                    with open(out_path, "wb") as f:
                        f.write(file_content.read()) 
                    saved.append(out_path)
                    print(f"Saved: {out_path}")
    if not saved:
        print("No generated files found in response annotations.")
    return saved

In [ ]:
# Example: try to fetch any files that response_2 created.
_ = download_generated_files(response_2)

## Suggested workshop prompts

Use these live during the workshop:

1. *Which factors seem most associated with higher reported stress in each dataset?*  
2. *Create a side-by-side chart comparing sleep-related variables across both datasets.*  
3. *Which occupation groups appear to show weaker work interest or poorer sleep outcomes?*  
4. *Generate a PNG chart and a CSV summary table for me to download.*

## Cleanup

Delete cloud resources created in the demo when you finish to avoid ongoing charges.

In [ ]:
# Uncomment when you are done.
# project.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
# print("Deleted agent version.")